In [1]:
import os, re, glob
import numpy as np, pandas as pd, pydicom
from concurrent.futures import ThreadPoolExecutor

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

def center_x(path):
    """Posisi x (LPS, mm) titik tengah citra pertama; x > 0 = sisi KIRI pasien."""
    f = sorted(glob.glob(path + "/*.dcm"))
    if not f:
        return np.nan, ""
    d = pydicom.dcmread(f[0], stop_before_pixels=True)
    try:
        ipp = np.array(d.ImagePositionPatient, float); iop = np.array(d.ImageOrientationPatient, float)
        ps = np.array(d.PixelSpacing, float)
        c = ipp + iop[:3] * ps[1] * d.Columns / 2 + iop[3:] * ps[0] * d.Rows / 2
        return float(c[0]), str(getattr(d, "Laterality", "") or "").strip().upper()
    except Exception:
        return np.nan, str(getattr(d, "Laterality", "") or "").strip().upper()

def study_side(args):
    split, uid, g = args
    xs, tags = [], []
    for s in g[g["Anatomical_Plane"].isin(["Coronal", "Axial"])]["SeriesInstanceUID"]:
        x, t = center_x(f"{COMP}/{split}_series/{uid}/{s}")
        if not np.isnan(x): xs.append(x)
        if t in ("L", "R"): tags.append(t)
    cx = float(np.median(xs)) if xs else np.nan
    if not np.isnan(cx) and abs(cx) > 20:
        side, src = ("L" if cx > 0 else "R"), "geometri"
    elif tags:
        side, src = max(set(tags), key=tags.count), "tag"
    elif not np.isnan(cx):
        side, src = ("L" if cx > 0 else "R"), "geometri-lemah"
    else:
        side, src = "R", "default"
    return {"StudyInstanceUID": uid, "split": split, "cx": cx, "side": side, "source": src,
            "tag": tags[0] if tags else ""}

jobs = []
for split in ["train", "test"]:
    ser = pd.read_csv(f"{COMP}/{split}_series.csv")
    jobs += [(split, uid, g) for uid, g in ser.groupby("StudyInstanceUID")]
with ThreadPoolExecutor(8) as ex:
    res = pd.DataFrame(list(ex.map(study_side, jobs)))

res.to_csv("/kaggle/working/knee_side.csv", index=False)
print("Studi:", res.groupby("split").size().to_dict())
print("Sisi (latih):", res[res.split == "train"]["side"].value_counts().to_dict())
print("Sumber penentuan:", res["source"].value_counts().to_dict())
both = res[res["tag"].isin(["L", "R"]) & (res["source"] == "geometri")]
print(f"Geometri vs tag DICOM: {len(both)} studi | cocok {(both['side'] == both['tag']).mean():.1%}")
print(res[res.split == "test"])

Studi: {'test': 3, 'train': 4407}
Sisi (latih): {'R': 2337, 'L': 2070}
Sumber penentuan: {'geometri': 4263, 'geometri-lemah': 76, 'tag': 71}
Geometri vs tag DICOM: 2113 studi | cocok 98.7%
                                       StudyInstanceUID split          cx  \
4407  1.2.826.0.1.3680043.8.498.10047035057544427318...  test   82.798719   
4408  1.2.826.0.1.3680043.8.498.10062861783145312629...  test  -89.538196   
4409  1.2.826.0.1.3680043.8.498.10067514707072572280...  test -101.743444   

     side    source tag  
4407    L  geometri   L  
4408    R  geometri      
4409    R  geometri      
